# 21-06 · Уничтожаем врагов и корабль

Практика к разделам [«Уничтожаем врагов»](../../site/chapters/glava-21/21-06-unichtozhenie.html) и [«Перерисовываем врагов. Игра окончена!»](../../site/chapters/glava-21/21-07-game-over.html).

## Цель

Проверить, что попадание пули уничтожает врага и начисляет очки, а враг, долетевший до низа экрана, завершает игру.

## Рабочий пример — попадание пули по врагу

In [1]:
import random

import pygame

SHIRINA, VYSOTA = 500, 600
FPS = 60

KORABL_SHIRINA, KORABL_VYSOTA = 50, 40
KORABL_SKOROST = 6

PULYA_SHIRINA, PULYA_VYSOTA = 4, 12
PULYA_SKOROST = 9

VRAG_SHIRINA, VRAG_VYSOTA = 40, 30
VRAG_SKOROST = 2
INTERVAL_POYAVLENIYA_VRAGA = 45

BELYJ = (255, 255, 255)
CHERNYJ = (10, 10, 20)
ZELYONYJ = (80, 220, 120)
KRASNYJ = (230, 60, 60)
ZHYOLTYJ = (240, 220, 80)

pygame.init()
screen = pygame.display.set_mode((SHIRINA, VYSOTA))
pygame.display.set_caption("Космический шутер")
clock = pygame.time.Clock()
shrift = pygame.font.SysFont(None, 32)
shrift_bolshoj = pygame.font.SysFont(None, 64)


def novaya_igra():
    return {
        "korabl": pygame.Rect(
            SHIRINA // 2 - KORABL_SHIRINA // 2,
            VYSOTA - KORABL_VYSOTA - 20,
            KORABL_SHIRINA,
            KORABL_VYSOTA,
        ),
        "puli": [],
        "vragi": [],
        "schet": 0,
        "kadrov_do_vraga": INTERVAL_POYAVLENIYA_VRAGA,
        "igra_okonchena": False,
    }


def obrabotat_klavishi(state, klavishi):
    korabl = state["korabl"]
    if klavishi[pygame.K_LEFT]:
        korabl.x -= KORABL_SKOROST
    if klavishi[pygame.K_RIGHT]:
        korabl.x += KORABL_SKOROST
    korabl.x = max(0, min(korabl.x, SHIRINA - KORABL_SHIRINA))


def vystrelit(state):
    korabl = state["korabl"]
    pulya = pygame.Rect(
        korabl.centerx - PULYA_SHIRINA // 2,
        korabl.top,
        PULYA_SHIRINA,
        PULYA_VYSOTA,
    )
    state["puli"].append(pulya)


def sozdat_vraga():
    x = random.randint(0, SHIRINA - VRAG_SHIRINA)
    return pygame.Rect(x, -VRAG_VYSOTA, VRAG_SHIRINA, VRAG_VYSOTA)


def obnovit_igru(state):
    if state["igra_okonchena"]:
        return

    for pulya in state["puli"]:
        pulya.y -= PULYA_SKOROST
    state["puli"] = [p for p in state["puli"] if p.bottom > 0]

    state["kadrov_do_vraga"] -= 1
    if state["kadrov_do_vraga"] <= 0:
        state["vragi"].append(sozdat_vraga())
        state["kadrov_do_vraga"] = INTERVAL_POYAVLENIYA_VRAGA

    for vrag in state["vragi"]:
        vrag.y += VRAG_SKOROST

    novye_puli = []
    novye_vragi = list(state["vragi"])
    for pulya in state["puli"]:
        popala = False
        for vrag in list(novye_vragi):
            if pulya.colliderect(vrag):
                novye_vragi.remove(vrag)
                state["schet"] += 10
                popala = True
                break
        if not popala:
            novye_puli.append(pulya)
    state["puli"] = novye_puli
    state["vragi"] = novye_vragi

    for vrag in state["vragi"]:
        if vrag.bottom >= VYSOTA or vrag.colliderect(state["korabl"]):
            state["igra_okonchena"] = True
            break


def narisovat(state):
    screen.fill(CHERNYJ)
    pygame.draw.rect(screen, ZELYONYJ, state["korabl"])
    for pulya in state["puli"]:
        pygame.draw.rect(screen, ZHYOLTYJ, pulya)
    for vrag in state["vragi"]:
        pygame.draw.rect(screen, KRASNYJ, vrag)

    tablo = shrift.render(f"Счёт: {state['schet']}", True, BELYJ)
    screen.blit(tablo, (10, 10))

    if state["igra_okonchena"]:
        nadpis = shrift_bolshoj.render("ИГРА ОКОНЧЕНА", True, BELYJ)
        rect = nadpis.get_rect(center=(SHIRINA // 2, VYSOTA // 2))
        screen.blit(nadpis, rect)

    pygame.display.flip()

pygame-ce 2.5.8 (SDL 2.32.10, Python 3.14.6)


In [2]:
state = novaya_igra()

# ставим врага прямо перед носом корабля и стреляем
vrag = pygame.Rect(state["korabl"].centerx - 20, state["korabl"].top - 30, VRAG_SHIRINA, VRAG_VYSOTA)
state["vragi"] = [vrag]
vystrelit(state)

for kadr in range(10):
    obnovit_igru(state)
    if state["schet"] > 0:
        break

print("Счёт:", state["schet"])
print("Врагов осталось:", len(state["vragi"]))

Счёт: 10
Врагов осталось: 0


## Проверка результата

In [3]:
assert state["schet"] == 10, "попадание должно добавить 10 очков"
assert len(state["vragi"]) == 0, "уничтоженный враг должен исчезнуть из списка"
print("Верно: враг уничтожен пулей, счёт увеличен на 10.")

Верно: враг уничтожен пулей, счёт увеличен на 10.


## Эксперимент — враг долетает до низа экрана

In [4]:
state2 = novaya_igra()
state2["vragi"] = [pygame.Rect(100, VYSOTA - VRAG_VYSOTA - 1, VRAG_SHIRINA, VRAG_VYSOTA)]

obnovit_igru(state2)
narisovat(state2)

print("Игра окончена:", state2["igra_okonchena"])
assert state2["igra_okonchena"] is True
print("Верно: враг, долетевший до низа экрана, завершает игру.")

Игра окончена: True
Верно: враг, долетевший до низа экрана, завершает игру.


## Эксперимент — обновление игры останавливается после конца игры

In [5]:
schet_do = state2["schet"]
vragov_do = len(state2["vragi"])

for kadr in range(20):
    obnovit_igru(state2)

assert state2["schet"] == schet_do
assert len(state2["vragi"]) == vragov_do
print("Верно: после игра окончена состояние больше не меняется.")

Верно: после игра окончена состояние больше не меняется.
